In [ ]:
import os
from langchain_community.document_loaders import TextLoader,DirectoryLoader
from dotenv import load_dotenv
load_dotenv()

True


STEP 1: Extracting Text from .txt files

In [ ]:

def load_document(document_path):

    #1.Check if docs directory exists
    if not os.path.exists(document_path):
        raise FileNotFoundError(f"The directory {document_path} does not exist.")

    #2.Load all .txt files from the docs directory - create loader object from DirectoryLoader class
    loader = DirectoryLoader(
        path=document_path,
        glob="*.txt",
        loader_cls=TextLoader,
        loader_kwargs={"encoding": "utf-8"}
    )
    
    documents = loader.load()
    
    #3.documents objects list empty or not
    if len(documents) == 0:
        raise FileNotFoundError(f"No .txt files found in {document_path}.") 
    
   #4.print first two documents from documents object list
    for i, doc in enumerate(documents[:2]):
        print(f"\nDocument {i+1}:")
        print(f"  Source: {doc.metadata['source']}")
        print(f"  Content length: {len(doc.page_content)} characters")
        print(f"  Content preview: {doc.page_content[:100]}...")
        print(f"  metadata: {doc.metadata}")

    return documents

STEP 2: Splitting the Documents into CHUNKS

In [13]:
from langchain_text_splitters import CharacterTextSplitter

def split_document(documents,chunk_size = 1000, chunk_overlap = 50):
    
    1.#create text_splitter object from CharacterTextSplitter class
    text_splitter = CharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunks = text_splitter.split_documents(documents)

    #2.print first two chunks from chunks object list
    for i, chunk in enumerate(chunks[:5]):
            print(f"\n--- Chunk {i+1} ---")
            print(f"Source: {chunk.metadata['source']}")
            print(f"Length: {len(chunk.page_content)} characters")
            print(f"Content:")
            print(chunk.page_content)
            print("-" * 50)

    return chunks


STEP 3: Create and Store Embeddings in Vector Store

In [14]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma 

def create_vectorstore(chunks, persist_directory):

    #Initialize embedding model - This model converts text chunks into numerical vectors (embeddings)  
    embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

    #Create vector store from document chunks - Converts each chunk into embeddings - Stores embeddings + metadata into ChromaDB - Saves (persists) the database to disk for reuse
    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embedding_model,
        persist_directory=persist_directory, 
        collection_metadata={"hnsw:space": "cosine"}
    )

    return vectorstore

STEP 4: Build workflow for extract document -> create chunks -> Store embedded chunks inside the vector database

In [15]:
def main():
    
    #1.Define paths
    document_path = "docs"
    persist_directory = "db/chroma_db"
    
    #2.Check if vector store already exists
    if os.path.exists(persist_directory):
        print("Vector store already exists. No need to re-process documents.")
        
        embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
        vectorstore = Chroma(
            persist_directory=persist_directory,
            embedding_function=embedding_model, 
            collection_metadata={"hnsw:space": "cosine"}
        )
        return vectorstore
    
    else:
        print("Persistent directory does not exist. Initializing vector store\n")
    
        # Step 1: Load documents
        documents = load_document(document_path)  

        # Step 2: Split into chunks
        chunks = split_document(documents)
    
        # Step 3: Create vector store
        vectorstore = create_vectorstore(chunks, persist_directory)
    
        print("\n Ingestion complete! Your documents are now ready for RAG queries.")
        return vectorstore

    
   


STEP 5: Run main function

In [17]:
if __name__ == "__main__":
    main()

Vector store already exists. No need to re-process documents.


C:\Users\gihan\AppData\Local\Temp\ipykernel_5264\3051393585.py:12: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(
